# Cálculo de α_c (C_crítico) para MNIST - Usando librería AKOrN

Este notebook calcula el valor crítico de acoplamiento para todas las imágenes del dataset MNIST.

**Optimizado para GPU de Google Colab usando la librería AKOrN del repositorio**

## 0. Montar Google Drive

In [ ]:
# Montar Google Drive para persistencia de datos
from google.colab import drive
import os

# Inicializar DRIVE_PATH con valor por defecto
DRIVE_PATH = '.'

print("⚠️  IMPORTANTE: Debes autorizar el acceso a Google Drive")
print("   1. Haz clic en el enlace que aparecerá abajo")
print("   2. Selecciona tu cuenta de Google")
print("   3. Autoriza el acceso a Drive")
print("   4. Copia el código de autorización")
print("   5. Pégalo cuando se te solicite\n")

try:
    # Intentar montar con timeout más largo (2 minutos)
    drive.mount('/content/drive', timeout_ms=120000)
    
    # Crear directorio para resultados si no existe
    DRIVE_PATH = '/content/drive/MyDrive/MNIST_C_Critical_AKOrN'
    os.makedirs(DRIVE_PATH, exist_ok=True)
    
    print(f"✅ Google Drive montado exitosamente")
    print(f"📁 Los resultados se guardarán en: {DRIVE_PATH}")
    print(f"   Sobrevivirán desconexiones y podrás acceder desde cualquier dispositivo")
except Exception as e:
    print(f"\n❌ Error al montar Google Drive: {e}")
    print("\n💡 Soluciones:")
    print("   1. Reinicia el runtime: Runtime → Restart runtime")
    print("   2. Asegúrate de completar la autorización")
    print("   3. Intenta ejecutar esta celda nuevamente")
    print("\n⚠️  ALTERNATIVA: Continuar sin Drive (datos se perderán al desconectar)")
    
    respuesta = input("¿Deseas continuar SIN Google Drive? (s/n): ")
    if respuesta.lower() == 's':
        DRIVE_PATH = '.'
        print(f"⚠️  Usando directorio local: {DRIVE_PATH}")
        print("   ⚠️  Los datos NO sobrevivirán desconexiones de Colab")
    else:
        print("\n❌ Reinicia el runtime e intenta de nuevo")
        # El usuario puede decidir continuar ejecutando celdas posteriores

## 1. Clonar Repositorio e Instalar Dependencias

In [ ]:
import os
from pathlib import Path

# Clonar repositorio si no existe
if not Path('/content/Proyecto-Inv.-teorica.').exists():
    !git clone https://github.com/ACRCris/Proyecto-Inv.-teorica..git
    print("✅ Repositorio clonado")
else:
    print("✅ Repositorio ya existe")

# Cambiar a la rama mac
os.chdir('/content/Proyecto-Inv.-teorica./codigo/akorn')
!git checkout mac

print(f"📂 Directorio actual: {os.getcwd()}")

In [ ]:
# Instalar dependencias
!pip install -q torch torchvision torchaudio
!pip install -q einops tqdm matplotlib

print("✅ Dependencias instaladas")

## 2. Importar Librerías y Módulos

In [ ]:
import sys
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.functional as TF
from torchvision import transforms

import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from tqdm.auto import tqdm
import random
from datetime import datetime
import json

# Añadir path del repositorio
sys.path.insert(0, '/content/Proyecto-Inv.-teorica./codigo/akorn')

# Importar módulos de AKOrN
from source.models.kuramoto import KBlock

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

print("\n✅ Módulos de AKOrN importados correctamente")

## 3. Configuración de Semillas para Reproducibilidad

In [ ]:
# Semilla fija para reproducibilidad
SEED = 1

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Semilla configurada: {SEED}")

## 4. Función para calcular el Parámetro de Orden

In [ ]:
def kuramoto_order_parameter(xs):
    """
    Calcula el parámetro de orden de Kuramoto r(t) para cada paso de tiempo.
    
    Args:
        xs: Lista de estados [T, B, C, H, W]
    
    Returns:
        np.array: Valores de r(t) para cada paso temporal
    """
    r_values = []

    for x in xs:
        # x: [B, C, H, W] donde C son las componentes del oscilador
        x = x[0]  # Tomamos primer batch

        # Calcula la fase de cada oscilador (asumiendo n=3, usar primeras 2 componentes)
        theta = torch.atan2(x[1], x[0])  # fase en radianes

        # Convierte a representación compleja e^{iθ}
        complex_phase = torch.exp(1j * theta)

        # Promedio global
        order_param = complex_phase.mean()

        # Magnitud = r(t)
        r = torch.abs(order_param)
        r_values.append(r.item())

    return np.array(r_values)

print("✅ Función de parámetro de orden definida")

## 5. Función para calcular C_crítico

In [ ]:
def calculate_c_critical(img, kblock, x_init, device, c_range, ch=3, h=64, w=64, T=30, gamma=0.7, del_t=0.9):
    """
    Calcula el valor crítico C_c para una imagen.
    
    Args:
        img: Imagen de entrada [1, H, W]
        kblock: Modelo KBlock
        x_init: Estado inicial [1, ch, h, w]
        device: Dispositivo (cuda/cpu)
        c_range: Rango de valores de C a evaluar
        ch, h, w: Dimensiones de los canales y resolución
        T: Pasos temporales
        gamma, del_t: Parámetros de integración
    
    Returns:
        dict: Diccionario con c_critical, R_final, y otros datos
    """
    # Preprocesar imagen
    img_resized = TF.resize(img, [h, w])
    img_channels = img_resized.repeat(ch, 1, 1)
    img_channels = img_channels.to(device)
    
    R_final = []
    
    # Evaluar para cada valor de C
    for c_val in c_range:
        x = x_init.clone().to(device)
        c = img_channels.unsqueeze(0) * c_val
        
        with torch.no_grad():
            x, xs, es = kblock(x, c, T=T, gamma=gamma, del_t=del_t, 
                              return_xs=True, return_es=True)
        
        R = kuramoto_order_parameter(xs)
        R_final.append(R[-1])
    
    R_final = np.array(R_final)
    
    # Calcular C_crítico como máximo de la derivada
    df = np.gradient(R_final, c_range)
    i_c = np.argmax(df)
    c_critical = c_range[i_c]
    
    return {
        'c_critical': float(c_critical),
        'R_final': R_final.tolist(),
        'c_range': c_range.tolist(),
        'max_derivative_idx': int(i_c)
    }

print("✅ Función de cálculo de C_crítico definida")

## 6. Configuración de Base de Datos SQLite

In [ ]:
def create_database(db_path='mnist_c_critical.db'):
    """
    Crea la base de datos SQLite con tablas para cada clase de MNIST.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Crear una tabla por cada clase (0-9)
    for clase in range(10):
        cursor.execute(f'''
        CREATE TABLE IF NOT EXISTS clase_{clase} (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            image_idx INTEGER UNIQUE NOT NULL,
            c_critical REAL NOT NULL,
            max_derivative_idx INTEGER,
            timestamp TEXT NOT NULL,
            seed INTEGER,
            n INTEGER,
            ch INTEGER,
            T INTEGER,
            gamma REAL,
            del_t REAL
        )
        ''')
    
    conn.commit()
    conn.close()
    print(f"✅ Base de datos creada: {db_path}")

def save_result(db_path, clase, image_idx, result, params):
    """
    Guarda un resultado en la base de datos.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    timestamp = datetime.now().isoformat()
    
    cursor.execute(f'''
    INSERT OR REPLACE INTO clase_{clase} 
    (image_idx, c_critical, max_derivative_idx, timestamp, seed, n, ch, T, gamma, del_t)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (image_idx, result['c_critical'], result['max_derivative_idx'], 
          timestamp, params['seed'], params['n'], params['ch'], 
          params['T'], params['gamma'], params['del_t']))
    
    conn.commit()
    conn.close()

def get_processed_images(db_path, clase):
    """
    Obtiene los índices de imágenes ya procesadas para una clase.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    try:
        cursor.execute(f'SELECT image_idx FROM clase_{clase}')
        processed = set(row[0] for row in cursor.fetchall())
    except sqlite3.OperationalError:
        # Tabla no existe aún
        processed = set()
    
    conn.close()
    return processed

print("✅ Funciones de base de datos definidas")

## 7. Cargar Dataset MNIST

In [ ]:
# Transformación básica (sin normalización para mantener valores originales)
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Descargar dataset
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, 
                                          download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, 
                                         download=True, transform=transform)

print(f"Train dataset: {len(train_dataset)} imágenes")
print(f"Test dataset: {len(test_dataset)} imágenes")

# Organizar por clases
imagenes_por_clase = {i: [] for i in range(10)}

for i in range(len(train_dataset)):
    _, label = train_dataset[i]
    imagenes_por_clase[label].append(i)

print("\nDistribución por clase:")
for clase in range(10):
    print(f"  Clase {clase}: {len(imagenes_por_clase[clase])} imágenes")

## 8. Configuración de Parámetros

In [ ]:
# Configuración del dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Parámetros del modelo (mismos que el notebook original)
PARAMS = {
    'seed': SEED,
    'ch': 3,
    'n': 3,
    'h': 64,
    'w': 64,
    'T': 30,
    'gamma': 0.7,
    'del_t': 0.9,
    'ksize': 3,
    'init_omg': 0.1,
}

# Rango de C a evaluar
c_range = np.arange(0.0, 2.0, 0.01)

# Base de datos - GUARDADA EN GOOGLE DRIVE
DB_PATH = os.path.join(DRIVE_PATH, 'mnist_c_critical_akorn.db')

print(f"\n⚙️ Parámetros configurados:")
for k, v in PARAMS.items():
    print(f"  {k}: {v}")
print(f"  c_range: {len(c_range)} valores de {c_range[0]:.2f} a {c_range[-1]:.2f}")
print(f"\n💾 Base de datos: {DB_PATH}")
print("  ✅ Los datos se guardarán en Google Drive y sobrevivirán desconexiones")

## 9. Inicializar Modelo y Estado Inicial

In [ ]:
# Reiniciar semilla para consistencia
set_seed(SEED)

# Crear modelo usando la librería AKOrN
kblock = KBlock(
    n=PARAMS['n'], 
    ch=PARAMS['ch'], 
    connectivity='conv', 
    T=PARAMS['T'], 
    ksize=PARAMS['ksize'],
    init_omg=PARAMS['init_omg'], 
    c_norm=None, 
    use_omega_c=False
).to(device)

# Estado inicial (se reutilizará para todas las imágenes)
x_init = torch.randn(1, PARAMS['ch'], PARAMS['h'], PARAMS['w'])

# Crear base de datos
create_database(DB_PATH)

print(f"✅ Modelo KBlock (AKOrN) inicializado en {device}")
print(f"✅ Estado inicial creado con forma {x_init.shape}")

## 10. Procesamiento del Dataset Completo

In [ ]:
# Seleccionar clases a procesar (puedes cambiar esto)
CLASES_A_PROCESAR = range(10)  # Todas las clases
# CLASES_A_PROCESAR = [0, 1]  # Solo clases 0 y 1 para pruebas

# Límite de imágenes por clase (None = todas)
LIMITE_POR_CLASE = None  # Cambiar a un número para pruebas rápidas, ej: 100

print(f"Procesando clases: {list(CLASES_A_PROCESAR)}")
if LIMITE_POR_CLASE:
    print(f"Límite por clase: {LIMITE_POR_CLASE} imágenes")
else:
    print("Procesando TODAS las imágenes de cada clase")

print("\n" + "="*60)
print("INICIANDO PROCESAMIENTO")
print("="*60 + "\n")

In [ ]:
# Procesar cada clase
resultados_resumen = {}

for clase in CLASES_A_PROCESAR:
    print(f"\n{'='*60}")
    print(f"PROCESANDO CLASE {clase}")
    print(f"{'='*60}")
    
    indices = imagenes_por_clase[clase]
    if LIMITE_POR_CLASE:
        indices = indices[:LIMITE_POR_CLASE]
    
    # 🔄 VERIFICAR IMÁGENES YA PROCESADAS
    processed_images = get_processed_images(DB_PATH, clase)
    indices_pendientes = [idx for idx in indices if idx not in processed_images]
    
    print(f"✅ Imágenes ya procesadas: {len(processed_images)}")
    print(f"⏳ Imágenes pendientes: {len(indices_pendientes)}")
    
    if len(indices_pendientes) == 0:
        print(f"✨ Clase {clase} ya está completamente procesada. Saltando...")
        # Cargar estadísticas existentes
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(f'SELECT c_critical FROM clase_{clase}')
        c_criticals = np.array([row[0] for row in cursor.fetchall()])
        conn.close()
    else:
        c_criticals = []
        
        # Barra de progreso
        for idx in tqdm(indices_pendientes, desc=f"Clase {clase}"):
            img, label = train_dataset[idx]
            
            # Calcular C_crítico
            result = calculate_c_critical(
                img, kblock, x_init, device, c_range,
                ch=PARAMS['ch'], h=PARAMS['h'], w=PARAMS['w'],
                T=PARAMS['T'], gamma=PARAMS['gamma'], del_t=PARAMS['del_t']
            )
            
            # Guardar en base de datos
            save_result(DB_PATH, clase, idx, result, PARAMS)
            
            c_criticals.append(result['c_critical'])
        
        # Cargar TODOS los resultados (incluyendo los previos)
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(f'SELECT c_critical FROM clase_{clase}')
        c_criticals = np.array([row[0] for row in cursor.fetchall()])
        conn.close()
    
    # Estadísticas
    resultados_resumen[clase] = {
        'n_images': len(c_criticals),
        'mean': float(np.mean(c_criticals)),
        'std': float(np.std(c_criticals)),
        'min': float(np.min(c_criticals)),
        'max': float(np.max(c_criticals)),
        'median': float(np.median(c_criticals))
    }
    
    print(f"\n📊 Estadísticas Clase {clase}:")
    print(f"  Imágenes procesadas: {resultados_resumen[clase]['n_images']}")
    print(f"  C_crítico medio: {resultados_resumen[clase]['mean']:.4f} ± {resultados_resumen[clase]['std']:.4f}")
    print(f"  Rango: [{resultados_resumen[clase]['min']:.4f}, {resultados_resumen[clase]['max']:.4f}]")
    print(f"  Mediana: {resultados_resumen[clase]['median']:.4f}")

print("\n" + "="*60)
print("PROCESAMIENTO COMPLETADO")
print("="*60)

## 11. Visualización de Resultados

In [ ]:
# Graficar distribución de C_crítico por clase
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for clase in CLASES_A_PROCESAR:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute(f'SELECT c_critical FROM clase_{clase}')
    c_vals = [row[0] for row in cursor.fetchall()]
    conn.close()
    
    axes[clase].hist(c_vals, bins=30, edgecolor='black', alpha=0.7)
    axes[clase].axvline(resultados_resumen[clase]['mean'], 
                       color='r', linestyle='--', linewidth=2,
                       label=f"Media: {resultados_resumen[clase]['mean']:.3f}")
    axes[clase].set_title(f"Clase {clase}")
    axes[clase].set_xlabel("C_crítico")
    axes[clase].set_ylabel("Frecuencia")
    axes[clase].legend()
    axes[clase].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(DRIVE_PATH, 'distribucion_c_critical_por_clase_akorn.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Gráfico guardado: {plot_path}")

In [ ]:
# Comparación de medias entre clases
clases = list(resultados_resumen.keys())
medias = [resultados_resumen[c]['mean'] for c in clases]
stds = [resultados_resumen[c]['std'] for c in clases]

plt.figure(figsize=(12, 6))
plt.errorbar(clases, medias, yerr=stds, fmt='o-', capsize=5, 
            markersize=8, linewidth=2, capthick=2)
plt.xlabel('Clase', fontsize=12)
plt.ylabel('C_crítico medio', fontsize=12)
plt.title('Comparación de C_crítico entre clases de MNIST (AKOrN)', fontsize=14)
plt.xticks(clases)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(DRIVE_PATH, 'comparacion_medias_clases_akorn.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Gráfico guardado: {plot_path}")

## 12. Descargar Resultados

In [ ]:
import json

# Guardar resumen en JSON (también en Drive)
json_path = os.path.join(DRIVE_PATH, 'resumen_c_critical_akorn.json')
with open(json_path, 'w') as f:
    json.dump(resultados_resumen, f, indent=2)

print("\n" + "="*60)
print("RESUMEN FINAL")
print("="*60)

for clase, stats in resultados_resumen.items():
    print(f"\nClase {clase}:")
    print(f"  N: {stats['n_images']}")
    print(f"  Media: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print(f"  Rango: [{stats['min']:.4f}, {stats['max']:.4f}]")
    print(f"  Mediana: {stats['median']:.4f}")

print(f"\n💾 Archivos guardados en Google Drive:")
print(f"  📊 Base de datos: {DB_PATH}")
print(f"  📄 Resumen JSON: {json_path}")
print(f"  📈 Gráficos: {DRIVE_PATH}/distribucion_*.png")
print(f"\n✅ Todos los archivos están en: MyDrive/MNIST_C_Critical_AKOrN/")
print(f"   Podrás acceder a ellos incluso después de desconectar Colab")
print(f"   Ubicación en Drive: https://drive.google.com/drive/my-drive")

print(f"\n🎉 Procesamiento completado exitosamente")

## 13. Test de Consistencia: Visualización de R_final vs C

In [ ]:
# Configuración del test
NUM_EJEMPLOS_POR_CLASE = 3  # Cuántas imágenes de cada clase mostrar
CLASES_TEST = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # Todas las clases

print("🧪 TEST DE CONSISTENCIA: R_final vs C")
print("="*60)
print(f"Visualizando {NUM_EJEMPLOS_POR_CLASE} ejemplos aleatorios por clase")
print(f"Clases a testear: {CLASES_TEST}")
print("="*60)

# Seleccionar ejemplos aleatorios de cada clase
np.random.seed(SEED)  # Para reproducibilidad
ejemplos_test = {}

for clase in CLASES_TEST:
    # Obtener imágenes procesadas de esta clase
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    try:
        cursor.execute(f'SELECT image_idx, c_critical FROM clase_{clase}')
        resultados = cursor.fetchall()
        
        if len(resultados) > 0:
            # Seleccionar aleatoriamente NUM_EJEMPLOS_POR_CLASE
            if len(resultados) >= NUM_EJEMPLOS_POR_CLASE:
                indices_seleccionados = np.random.choice(len(resultados), 
                                                        NUM_EJEMPLOS_POR_CLASE, 
                                                        replace=False)
            else:
                indices_seleccionados = range(len(resultados))
            
            ejemplos_test[clase] = [resultados[i] for i in indices_seleccionados]
        else:
            print(f"⚠️  Clase {clase}: No hay datos procesados")
            ejemplos_test[clase] = []
    except sqlite3.OperationalError:
        print(f"⚠️  Clase {clase}: Tabla no existe")
        ejemplos_test[clase] = []
    
    conn.close()

print("\n✅ Ejemplos seleccionados:")
for clase, ejemplos in ejemplos_test.items():
    if ejemplos:
        print(f"  Clase {clase}: {len(ejemplos)} imágenes")

In [ ]:
# Recalcular R_final vs C para los ejemplos seleccionados
datos_test = {}

print("\n🔄 Recalculando curvas R_final vs C...")

for clase, ejemplos in tqdm(ejemplos_test.items(), desc="Clases"):
    if not ejemplos:
        continue
    
    datos_test[clase] = []
    
    for image_idx, c_critical_guardado in ejemplos:
        img, label = train_dataset[image_idx]
        
        # Recalcular la curva completa
        result = calculate_c_critical(
            img, kblock, x_init, device, c_range,
            ch=PARAMS['ch'], h=PARAMS['h'], w=PARAMS['w'],
            T=PARAMS['T'], gamma=PARAMS['gamma'], del_t=PARAMS['del_t']
        )
        
        datos_test[clase].append({
            'image_idx': image_idx,
            'c_critical_guardado': c_critical_guardado,
            'c_critical_recalculado': result['c_critical'],
            'R_final': result['R_final'],
            'c_range': result['c_range'],
            'max_derivative_idx': result['max_derivative_idx']
        })

print("✅ Recálculo completado")

In [ ]:
# Visualización: Grid de gráficas R_final vs C
n_clases = len([c for c in CLASES_TEST if c in datos_test and datos_test[c]])
n_cols = 5
n_rows = int(np.ceil(n_clases / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_clases == 1 else axes

plot_idx = 0

for clase in CLASES_TEST:
    if clase not in datos_test or not datos_test[clase]:
        continue
    
    ax = axes[plot_idx]
    
    # Plotear las curvas de los ejemplos
    for i, dato in enumerate(datos_test[clase]):
        c_vals = dato['c_range']
        R_vals = dato['R_final']
        c_crit = dato['c_critical_recalculado']
        idx_crit = dato['max_derivative_idx']
        
        # Curva R_final vs C
        ax.plot(c_vals, R_vals, alpha=0.7, linewidth=1.5, 
               label=f"Img {dato['image_idx']}")
        
        # Marcar C_crítico
        ax.axvline(c_crit, color=f'C{i}', linestyle='--', 
                  linewidth=1, alpha=0.5)
        ax.plot(c_crit, R_vals[idx_crit], 'o', 
               color=f'C{i}', markersize=8)
    
    # Marcar la media de la clase
    if clase in resultados_resumen:
        media_clase = resultados_resumen[clase]['mean']
        ax.axvline(media_clase, color='red', linestyle='-', 
                  linewidth=2, label=f'Media clase: {media_clase:.3f}')
    
    ax.set_xlabel('C (Acoplamiento)', fontsize=10)
    ax.set_ylabel('R_final (Parámetro de orden)', fontsize=10)
    ax.set_title(f'Clase {clase}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    
    plot_idx += 1

# Ocultar ejes sobrantes
for idx in range(plot_idx, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
test_plot_path = os.path.join(DRIVE_PATH, 'test_consistencia_R_vs_C_akorn.png')
plt.savefig(test_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Gráfico de test guardado: {test_plot_path}")

In [ ]:
# Visualización detallada: Una imagen por clase con la derivada
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
axes = axes.flatten()

for clase_idx, clase in enumerate(CLASES_TEST):
    if clase not in datos_test or not datos_test[clase]:
        axes[clase_idx].text(0.5, 0.5, f'Clase {clase}\nSin datos', 
                            ha='center', va='center', fontsize=12)
        axes[clase_idx].axis('off')
        continue
    
    # Tomar el primer ejemplo
    dato = datos_test[clase][0]
    c_vals = np.array(dato['c_range'])
    R_vals = np.array(dato['R_final'])
    c_crit = dato['c_critical_recalculado']
    idx_crit = dato['max_derivative_idx']
    
    ax = axes[clase_idx]
    
    # Subplot principal: R_final vs C
    color_R = 'tab:blue'
    ax.plot(c_vals, R_vals, color=color_R, linewidth=2, label='R_final')
    ax.axvline(c_crit, color='red', linestyle='--', linewidth=2, 
              label=f'C_crítico = {c_crit:.3f}')
    ax.plot(c_crit, R_vals[idx_crit], 'ro', markersize=10, 
           label=f'R = {R_vals[idx_crit]:.3f}')
    ax.set_xlabel('C (Acoplamiento)', fontsize=10)
    ax.set_ylabel('R_final', fontsize=10, color=color_R)
    ax.tick_params(axis='y', labelcolor=color_R)
    ax.set_title(f'Clase {clase} - Imagen {dato["image_idx"]}', 
                fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Eje secundario: Derivada
    ax2 = ax.twinx()
    color_dR = 'tab:orange'
    dR_dc = np.gradient(R_vals, c_vals)
    ax2.plot(c_vals, dR_dc, color=color_dR, linewidth=1.5, 
            alpha=0.7, linestyle=':', label='dR/dC')
    ax2.axvline(c_crit, color='red', linestyle='--', linewidth=2, alpha=0.3)
    ax2.plot(c_crit, dR_dc[idx_crit], 's', color=color_dR, 
            markersize=8, label=f'max(dR/dC) = {dR_dc[idx_crit]:.3f}')
    ax2.set_ylabel('dR/dC', fontsize=10, color=color_dR)
    ax2.tick_params(axis='y', labelcolor=color_dR)
    
    # Leyenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='best', fontsize=8)

plt.tight_layout()
test_detail_path = os.path.join(DRIVE_PATH, 'test_detallado_R_derivada_akorn.png')
plt.savefig(test_detail_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Gráfico detallado guardado: {test_detail_path}")

In [ ]:
# Verificación de consistencia: Comparar valores guardados vs recalculados
print("\n" + "="*60)
print("VERIFICACIÓN DE CONSISTENCIA")
print("="*60)

inconsistencias = []

for clase in CLASES_TEST:
    if clase not in datos_test or not datos_test[clase]:
        continue
    
    print(f"\n📊 Clase {clase}:")
    
    for dato in datos_test[clase]:
        c_guardado = dato['c_critical_guardado']
        c_recalculado = dato['c_critical_recalculado']
        diferencia = abs(c_guardado - c_recalculado)
        
        # Tolerancia: 0.01 (el paso de c_range)
        es_consistente = diferencia < 0.02
        
        simbolo = "✅" if es_consistente else "⚠️"
        print(f"  {simbolo} Img {dato['image_idx']:>5}: "
              f"Guardado={c_guardado:.4f}, "
              f"Recalculado={c_recalculado:.4f}, "
              f"Δ={diferencia:.4f}")
        
        if not es_consistente:
            inconsistencias.append({
                'clase': clase,
                'image_idx': dato['image_idx'],
                'guardado': c_guardado,
                'recalculado': c_recalculado,
                'diferencia': diferencia
            })

if inconsistencias:
    print(f"\n⚠️  Se encontraron {len(inconsistencias)} inconsistencias:")
    for inc in inconsistencias:
        print(f"  - Clase {inc['clase']}, Img {inc['image_idx']}: Δ = {inc['diferencia']:.4f}")
else:
    print(f"\n✅ Todas las mediciones son consistentes (tolerancia: ±0.02)")

print("\n" + "="*60)
print("TEST DE CONSISTENCIA COMPLETADO")
print("="*60)

In [ ]:
# Visualizar las imágenes MNIST usadas en el test junto con sus curvas
n_ejemplos_mostrar = min(5, max([len(datos_test.get(c, [])) for c in CLASES_TEST]))

if n_ejemplos_mostrar > 0:
    fig = plt.figure(figsize=(20, 4 * n_ejemplos_mostrar))
    gs = fig.add_gridspec(n_ejemplos_mostrar, 12, hspace=0.4, wspace=0.5)
    
    ejemplo_idx = 0
    
    for clase in CLASES_TEST:
        if clase not in datos_test or not datos_test[clase]:
            continue
        
        if ejemplo_idx >= n_ejemplos_mostrar:
            break
        
        dato = datos_test[clase][0]
        img, label = train_dataset[dato['image_idx']]
        
        # Subplot 1: Imagen MNIST
        ax_img = fig.add_subplot(gs[ejemplo_idx, :2])
        ax_img.imshow(img.squeeze(), cmap='gray')
        ax_img.set_title(f'Clase {clase}\nImg {dato["image_idx"]}', 
                        fontsize=10, fontweight='bold')
        ax_img.axis('off')
        
        # Subplot 2: Curva R_final vs C
        ax_curva = fig.add_subplot(gs[ejemplo_idx, 2:7])
        c_vals = np.array(dato['c_range'])
        R_vals = np.array(dato['R_final'])
        c_crit = dato['c_critical_recalculado']
        idx_crit = dato['max_derivative_idx']
        
        ax_curva.plot(c_vals, R_vals, 'b-', linewidth=2, label='R_final')
        ax_curva.axvline(c_crit, color='red', linestyle='--', linewidth=2)
        ax_curva.plot(c_crit, R_vals[idx_crit], 'ro', markersize=10)
        ax_curva.set_xlabel('C', fontsize=9)
        ax_curva.set_ylabel('R_final', fontsize=9)
        ax_curva.set_title(f'Transición de sincronización', fontsize=10)
        ax_curva.grid(True, alpha=0.3)
        ax_curva.legend(fontsize=8)
        
        # Subplot 3: Derivada dR/dC
        ax_deriv = fig.add_subplot(gs[ejemplo_idx, 7:])
        dR_dc = np.gradient(R_vals, c_vals)
        ax_deriv.plot(c_vals, dR_dc, 'orange', linewidth=2, label='dR/dC')
        ax_deriv.axvline(c_crit, color='red', linestyle='--', linewidth=2, 
                        label=f'C_crítico = {c_crit:.3f}')
        ax_deriv.plot(c_crit, dR_dc[idx_crit], 's', color='red', markersize=10)
        ax_deriv.set_xlabel('C', fontsize=9)
        ax_deriv.set_ylabel('dR/dC', fontsize=9)
        ax_deriv.set_title(f'Derivada (max en C_crítico)', fontsize=10)
        ax_deriv.grid(True, alpha=0.3)
        ax_deriv.legend(fontsize=8)
        
        ejemplo_idx += 1
    
    test_combined_path = os.path.join(DRIVE_PATH, 'test_imagenes_curvas_akorn.png')
    plt.savefig(test_combined_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Visualización combinada guardada: {test_combined_path}")
else:
    print("⚠️  No hay ejemplos para mostrar")